In [1]:
pip install torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn.functional as F

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])   
w1 = torch.tensor([2.2])   
b = torch.tensor([0.0])           
z = x1 * w1 + b                
a = torch.sigmoid(z)              
loss = F.binary_cross_entropy(a, y)

loss

tensor(0.0852)

#### **Computing gradient via autograd**

In [4]:
import torch.nn.functional as F
from torch.autograd import grad
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
z = x1 * w1 + b 
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)
grad_L_w1 = grad(loss, w1, retain_graph=True)  
grad_L_b = grad(loss, b, retain_graph=True)

In [5]:
print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [6]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


#### **A multilayer perceptron with two hidden layer**

In [7]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):   
        super().__init__()
        self.layers = torch.nn.Sequential(
            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),   
            torch.nn.ReLU(),              
            # 2nd hidden layer
            torch.nn.Linear(30, 20),   
            torch.nn.ReLU(),
             # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits

In [8]:
model = NeuralNetwork(50, 3)

In [9]:
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


In [10]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total number of trainable model parameters:", num_params)

Total number of trainable model parameters: 2213


In [11]:
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0600, -0.0186,  0.0960,  ..., -0.0288, -0.0407,  0.0863],
        [ 0.0790, -0.0464,  0.0440,  ..., -0.1034, -0.1118,  0.0539],
        [ 0.0510, -0.0534, -0.0958,  ...,  0.0190, -0.1263,  0.1266],
        ...,
        [-0.1138,  0.0856, -0.1278,  ...,  0.1224, -0.1113,  0.0683],
        [ 0.1406, -0.0970,  0.0680,  ...,  0.0243,  0.1180, -0.0412],
        [ 0.1231, -0.0489,  0.0415,  ..., -0.0692,  0.1357, -0.0728]],
       requires_grad=True)


In [12]:
print(model.layers[0].weight.shape)

torch.Size([30, 50])


In [13]:
torch.manual_seed(123)
model = NeuralNetwork(50, 3)
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


In [14]:
torch.manual_seed(123)
X = torch.rand((1, 50))
out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


In [15]:
with torch.no_grad():
    out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


In [16]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.3113, 0.3934, 0.2952]])
